# Train Essay Scoring Models

This notebook orchestrates the complete training pipeline for ASAP-AES essay scoring models.

**Workflow:**
1. Load ASAP-AES data from Excel files
2. Extract linguistic features for all essays
3. Perform stratified train/validation/test split per essay set
4. Train XGBoost regressors (one per essay set)
5. Optimize hyperparameters using Optuna (QWK metric)
6. Evaluate models on test sets
7. Save trained models and scalers to disk
8. Generate feature importance analysis

**Expected Runtime:** 30-60 minutes depending on dataset size and Optuna trials
**Requirements:** Completion of ASAP-AES_EDA.ipynb (or data in `../data/`)

In [ ]:
# Setup and Imports
# =================

import os
import sys
import json
import time
import pickle
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

# Import custom modules
try:
    from data_loader import ASAPDataLoader
    from feature_extractor import EssayFeatureExtractor
    from essay_score_regressor import EssayScoreRegressor
    from metrics import compute_qwk, evaluate_predictions
    print("✓ All custom modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure you're running this notebook from Short_Answer_NLP/notebooks/")

# Configuration - Data consolidated in Unified_Datasets
DATA_DIR = Path('../../Unified_Datasets')
MODEL_DIR = Path('../models')
OUTPUT_DIR = Path('../outputs')
CONFIG_FILE = Path('../configs/essay_scoring_config.json')

# Create directories if needed
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Suppress warnings
warnings.filterwarnings('ignore')

print(f"✓ Configuration loaded")
print(f"  Data directory: {DATA_DIR.absolute()}")
print(f"  Model save directory: {MODEL_DIR.absolute()}")

In [ ]:
# Step 1: Load ASAP-AES Data
# ==========================

print("=" * 80)
print("STEP 1: LOAD ASAP-AES DATA")
print("=" * 80)

loader = ASAPDataLoader()

# Load all essay sets
essays_df, metadata = loader.load_essays_from_excel(DATA_DIR)

if essays_df is None or len(essays_df) == 0:
    print("❌ No data loaded. Check that ASAP-AES Excel files are in ../data/")
    print("Current files in data directory:")
    if DATA_DIR.exists():
        for f in DATA_DIR.glob('*'):
            print(f"  - {f.name}")
else:
    print(f"✓ Loaded {len(essays_df)} essays across {essays_df['essay_set'].nunique()} sets")
    print(f"\nEssay set summary:")
    for set_id in sorted(essays_df['essay_set'].unique()):
        set_data = essays_df[essays_df['essay_set'] == set_id]
        print(f"  Set {set_id}: {len(set_data)} essays")

In [ ]:
# Step 2: Extract Features
# ========================

print("\n" + "=" * 80)
print("STEP 2: EXTRACT LINGUISTIC FEATURES")
print("=" * 80)

if essays_df is not None and len(essays_df) > 0:
    # Initialize feature extractor
    print("Initializing feature extractor (loading spaCy model)...")
    extractor = EssayFeatureExtractor()
    
    # Extract features for all essays
    print(f"Extracting features for {len(essays_df)} essays...")
    start_time = time.time()
    
    features_df = extractor.batch_extract(essays_df['essay_text'].values)
    
    # Add essay_set and score to features
    features_df['essay_set'] = essays_df['essay_set'].values
    features_df['score'] = essays_df['score'].values
    
    elapsed = time.time() - start_time
    print(f"✓ Features extracted in {elapsed:.1f}s")
    print(f"  Features shape: {features_df.shape}")
    print(f"  Features: {list(features_df.columns[:-2])}")
    
    # Display sample
    print(f"\nFirst essay features:")
    print(features_df.iloc[0])
else:
    print("⚠️  No data available for feature extraction.")

In [ ]:
# Step 3: Train Models Per Essay Set
# ==================================

print("\n" + "=" * 80)
print("STEP 3: TRAIN XGBOOST MODELS PER ESSAY SET")
print("=" * 80)

if 'features_df' in locals() and features_df is not None:
    trained_models = {}
    training_results = []
    
    for set_id in sorted(features_df['essay_set'].unique()):
        print(f"\n📊 Training model for Essay Set {set_id}...")
        
        # Get data for this essay set
        set_data = features_df[features_df['essay_set'] == set_id].copy()
        print(f"   Essays: {len(set_data)}")
        
        # Prepare features and labels
        feature_cols = [col for col in set_data.columns if col not in ['essay_set', 'score']]
        X = set_data[feature_cols].values
        y = set_data['score'].values
        
        # Stratified split: 70% train, 15% val, 15% test
        X_train, X_test, y_train, y_test, X_val, y_val = loader.stratified_split(
            X, y, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42
        )
        
        print(f"   Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Train regressor with Optuna hyperparameter tuning
        regressor = EssayScoreRegressor(essay_set=set_id, n_trials=50)
        
        start_time = time.time()
        regressor.train(X_train, y_train, X_val, y_val, optimize_for_qwk=True)
        elapsed = time.time() - start_time
        
        print(f"   Training completed in {elapsed:.1f}s")
        
        # Evaluate on test set
        y_pred_test = regressor.model.predict(X_test)
        qwk_test = compute_qwk(y_test.astype(int), np.round(y_pred_test).astype(int))
        mae_test = np.mean(np.abs(y_test - y_pred_test))
        rmse_test = np.sqrt(np.mean((y_test - y_pred_test) ** 2))
        
        print(f"   Test QWK: {qwk_test:.4f} | MAE: {mae_test:.3f} | RMSE: {rmse_test:.3f}")
        
        # Save model
        model_path = MODEL_DIR / f'essay_set_{set_id}_model.pkl'
        regressor.save_model(model_path)
        print(f"   Model saved to {model_path.name}")
        
        trained_models[set_id] = regressor
        training_results.append({
            'Essay Set': set_id,
            'Train Essays': len(X_train),
            'Test Essays': len(X_test),
            'QWK (Test)': f"{qwk_test:.4f}",
            'MAE': f"{mae_test:.3f}",
            'RMSE': f"{rmse_test:.3f}"
        })
    
    # Summary table
    print("\n" + "=" * 80)
    print("TRAINING SUMMARY")
    results_df = pd.DataFrame(training_results)
    print(results_df.to_string(index=False))
    
    # Save summary
    results_df.to_csv(OUTPUT_DIR / 'training_results.csv', index=False)
    print(f"\n✓ Results saved to {OUTPUT_DIR / 'training_results.csv'}")
else:
    print("⚠️  No features available for training.")

In [ ]:
# Step 4: Feature Importance Analysis
# ===================================

print("\n" + "=" * 80)
print("STEP 4: FEATURE IMPORTANCE")
print("=" * 80)

if 'trained_models' in locals():
    # Get feature names
    feature_cols = [col for col in features_df.columns if col not in ['essay_set', 'score']]
    
    for set_id, regressor in sorted(trained_models.items()):
        print(f"\n📈 Top features for Essay Set {set_id}:")
        
        # Get feature importance
        importance = regressor.get_feature_importance()
        
        # Create sorted importance dict
        importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)
        
        for i, (feature, score) in enumerate(importance_sorted[:10], 1):
            print(f"   {i:2d}. {feature:30s}: {score:.4f}")
    
    # Plot feature importance for first model
    if len(trained_models) > 0:
        first_set_id = sorted(trained_models.keys())[0]
        first_model = trained_models[first_set_id]
        
        importance = first_model.get_feature_importance()
        importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        features, scores = zip(*importance_sorted)
        ax.barh(features, scores, color='steelblue')
        ax.set_xlabel('Importance Score', fontsize=11)
        ax.set_title(f'Top 10 Features - Essay Set {first_set_id}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print(f"\n✓ Feature importance plot displayed")
else:
    print("⚠️  No trained models available for feature analysis.")

In [ ]:
# Summary & Next Steps
# ====================

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

if 'trained_models' in locals() and trained_models:
    print(f"\n✓ Trained {len(trained_models)} models successfully")
    print(f"✓ Models saved to: {MODEL_DIR.absolute()}")
    print(f"✓ Results saved to: {OUTPUT_DIR / 'training_results.csv'}")
    
    print("\n📌 Model Performance Summary:")
    if 'results_df' in locals():
        print(results_df.to_string(index=False))
    
    print("\n" + "=" * 80)
    print("NEXT STEPS:")
    print("=" * 80)
    print("\n1. Review training results in training_results.csv")
    print("2. If QWK scores < 0.5, consider:")
    print("   - Re-tuning hyperparameters (increase n_trials)")
    print("   - Adding more features")
    print("   - Checking data quality issues")
    print("\n3. Run 'Evaluate_Essays.ipynb' to:")
    print("   - Load saved models")
    print("   - Score new essays interactively")
    print("   - Generate personalized feedback")
    print("\n4. Deploy models via Flask API (Phase 2)")
    
    print("\n" + "=" * 80)
    print(f"✓ Timestamp: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("\n⚠️  No models trained. Check previous steps for errors.")